In [9]:
from util import *

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
import pickle

from scipy import stats
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.decomposition import PCA
from joblib import dump, load

In [10]:
data_log, data_dash = get_data_frames()

In [11]:
features

,ingress_global_timestamp3,egress_global_timestamp3,enq_timestamp3,enq_qdepth3,deq_timedelta3,deq_qdepth3,ingress_global_timestamp2,egress_global_timestamp2,enq_timestamp2,enq_qdepth2,deq_timedelta2,deq_qdepth2,ingress_global_timestamp1,egress_global_timestamp1,enq_timestamp1,enq_qdepth1,deq_timedelta1,deq_qdepth1
timestamp,,,,,,,,,,,,,,,,,,
1621899282,3.909507e+10,3.909507e+10,4.403598e+08,0.00000,34.000000,0.000000,3.916932e+10,3.916932e+10,5.146125e+08,0.000000,34.000000,0.000000,3.924525e+10,3.924525e+10,5.905407e+08,0.000000,32.400000,0.000000
1621899283,3.909580e+10,3.909580e+10,4.410972e+08,0.00000,26.333333,0.000000,3.917006e+10,3.917006e+10,5.153497e+08,0.000000,36.500000,0.000000,3.924598e+10,3.924598e+10,5.912778e+08,0.000000,30.333333,0.000000
1621899284,3.909691e+10,3.909691e+10,4.422026e+08,0.00000,27.166667,0.000000,3.917116e+10,3.917116e+10,5.164553e+08,0.000000,26.000000,0.000000,3.924709e+10,3.924709e+10,5.923834e+08,0.000000,31.166667,0.000000
1621899285,3.909790e+10,3.909790e+10,4.431917e+08,0.00000,28.142857,0.000000,3.917215e+10,3.917215e+10,5.174444e+08,0.000000,31.428571,0.000000,3.924808e+10,3.924808e+10,5.933725e+08,0.000000,28.285714,0.000000
1621899286,3.909885e+10,3.909885e+10,4.441475e+08,0.00000,41.888889,0.000000,3.917311e+10,3.917311e+10,5.184002e+08,0.000000,31.777778,0.000000,3.924903e+10,3.924903e+10,5.943283e+08,0.000000,32.666667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1621927359,5.522485e+10,5.522485e+10,2.106121e+09,0.00764,120.203704,0.008169,5.529924e+10,5.529924e+10,2.111009e+09,0.757718,718.065258,0.572315,5.537510e+10,5.537510e+10,2.116211e+09,1.394736,959.173138,1.078428
1621931540,5.522485e+10,5.522485e+10,2.106121e+09,0.00764,120.203704,0.008169,5.529924e+10,5.529924e+10,2.111009e+09,0.757718,718.065258,0.572315,5.537510e+10,5.537510e+10,2.116211e+09,1.394736,959.173138,1.078428
1621931541,5.522485e+10,5.522485e+10,2.106121e+09,0.00764,120.203704,0.008169,5.529924e+10,5.529924e+10,2.111009e+09,0.757718,718.065258,0.572315,5.537510e+10,5.537510e+10,2.116211e+09,1.394736,959.173138,1.078428


In [14]:
features, labels = get_data_concated(data_log, data_dash)
mae, nmae, rf_model = train_rf_model_gridsearch(features, labels)
print(f"{nmae*100}%")
print(xgb_model.feature_importances_)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV 1/5] END bootstrap=False, max_depth=55, max_features=auto, min_samples_leaf=4, min_samples_split=5, n_estimators=100;, score=nan total time=   0.0s
[CV 2/5] END bootstrap=False, max_depth=55, max_features=auto, min_samples_leaf=4, min_samples_split=5, n_estimators=100;, score=nan total time=   0.0s
[CV 3/5] END bootstrap=False, max_depth=55, max_features=auto, min_samples_leaf=4, min_samples_split=5, n_estimators=100;, score=nan total time=   0.0s
[CV 4/5] END bootstrap=False, max_depth=55, max_features=auto, min_samples_leaf=4, min_samples_split=5, n_estimators=100;, score=nan total time=   0.0s
[CV 5/5] END bootstrap=False, max_depth=55, max_features=auto, min_samples_leaf=4, min_samples_split=5, n_estimators=100;, score=nan total time=   0.0s
[CV 1/5] END bootstrap=False, max_depth=10, max_features=auto, min_samples_leaf=4, min_samples_split=2, n_estimators=10;, score=nan total time=   0.0s
[CV 2/5] END bootstrap=Fal

KeyboardInterrupt: 

In [13]:
def get_data_concated(data_log, data_dash):
    
    for colum in data_log.columns:
        if(data_log[colum].std() == 0.0):
            data_log = data_log.drop(columns=[colum])

    result_total = pd.merge(data_log, data_dash, on='timestamp', how='left')
    result_total = result_total.fillna(result_total.mean())

    z_scores = np.abs(stats.zscore(result_total['framesDisplayedCalc']))
    threshold = 3
    result_total = result_total[(z_scores < threshold)]

    labels = result_total['framesDisplayedCalc']

    columns_to_remove = list(data_dash.columns)
    columns_to_remove.remove('timestamp')
    result_total = result_total.drop(columns=columns_to_remove)
    return result_total, labels

In [8]:
def train_xgb_model(data, answer):
    X_train, X_validation, y_train, y_validation = train_test_split(
        data,
        np.ravel(answer),
        test_size=0.20,
        random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_validation_scaled = scaler.transform(X_validation)

    xgb_model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.01,  
        max_depth=20, 
        n_jobs=2,
        random_state=42
    )

    mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    cv_scores = cross_val_score(
        xgb_model,
        X_train_scaled,
        y_train,
        cv=kf,
        scoring=mae_scorer
    )
    avg_cv_score = np.mean(cv_scores)

    xgb_model.fit(
        X_train_scaled,
        y_train,
        eval_set=[(X_validation_scaled, y_validation)],
        verbose=True
    )

    y_pred = xgb_model.predict(X_validation_scaled)
    mae = mean_absolute_error(y_validation, y_pred)
    nmae = mae / np.mean(y_validation)

    return mae, nmae, xgb_model

In [8]:
def nmae(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    nmae_value = mae / np.mean(y_true)
    return nmae_value
def train_rf_model_gridsearch(features_train: pd.DataFrame, labels_train: pd.Series):
    X_train, X_validation, y_train, y_validation = train_test_split(
        features_train,
        np.ravel(labels_train),
        test_size=0.20,
        random_state=42
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_validation_scaled = scaler.transform(X_validation)

    rf_model = RandomForestRegressor(random_state=42, n_jobs=2)

    nmae_scorer = make_scorer(nmae, greater_is_better=False)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    n_estimators = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150]
    max_features = ['auto', 'sqrt']
    max_depth = [int(x) for x in np.linspace(start=10, stop=100, num=11)]
    max_depth.append(None)
    min_samples_split = [2, 5, 10]
    min_samples_leaf = [1, 2, 4]
    bootstrap = [True, False]

    random_grid = {
        'n_estimators': n_estimators,
        'max_features': max_features,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'bootstrap': bootstrap
    }

    grid_search = RandomizedSearchCV(
        estimator=rf_model,
        param_distributions=random_grid,
        scoring=nmae_scorer,
        cv=kf,
        n_iter=100, 
        verbose=3,  
        n_jobs=2
    )

    grid_search.fit(X_train_scaled, y_train)

    best_rf_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    print(f"Melhores hiperparâmetros: {best_params}")

    y_pred_rf = best_rf_model.predict(X_validation_scaled)
    mae_rf = mean_absolute_error(y_validation, y_pred_rf)
    nmae_rf = nmae(y_validation, y_pred_rf)

    feature_importances = best_rf_model.feature_importances_
    print(f"Importâncias das features: {feature_importances}")

    return mae_rf, nmae_rf, best_rf_model
